# XGBoost Three-Class AQI: Pollutants Only vs Pollutants + Weather

Test whether today's weather summaries add predictive information beyond today's pollutant readings for next-day **three-class** AQI prediction. Compare XGBoost using six pollutant inputs alone with XGBoost using those pollutants plus nine weather summaries.

The notebook reads `air_quality.aqi_daily` only. It does not write to MySQL or save files.

## Evaluation design

- Rebuild AQI and exact next-calendar-day targets with the shared AQI labeling rules.
- Use one chronological split by unique date: the first 80% for training and the last 20% for the untouched test period. All stations for a date stay in the same split.
- Tune XGBoost only on the training dates using four expanding-window date-based folds and macro-F1.
- Compare accuracy, macro-F1, weighted F1, and balanced accuracy with majority and persistence baselines.
- Review per-class precision, recall, F1, support, and confusion matrices.

Both three-class models receive the same pollutant readings. The second model also receives weather summaries. Dates, splits, folds, target labels, tuning grid, and evaluation metrics are identical so the measured difference isolates the value of adding weather features within this experiment.

## 1. Imports and project settings

The notebook locates the shared AQI helper and reads from the configured MySQL `air_quality` database. Set `AIR_QUALITY_DB_URL` or `DB_URL` if needed.

In [1]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, make_scorer, precision_recall_fscore_support,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [HERE, *HERE.parents]
     if (p / "air-quality-classification" / "app" / "aqi_labels.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the Data_Science repository or one of its subfolders.")
APP_DIR = PROJECT_ROOT / "air-quality-classification" / "app"
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
from aqi_labels import O3_1H_BREAKPOINTS, POLLUTANT_BREAKPOINTS, interpolate_iaqi

DB_URL = os.environ.get(
    "AIR_QUALITY_DB_URL",
    os.environ.get("DB_URL", "mysql+pymysql://root:@localhost/air_quality"),
)
SOURCE_TABLE = "aqi_daily"
POLLUTANTS = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
WEATHER = [
    "TEMP_mean", "TEMP_range", "PRES_mean", "DEWP_mean", "RH_mean",
    "RAIN_sum", "WSPM_mean", "wd_sin", "wd_cos",
]
FULL_FEATURES = POLLUTANTS + WEATHER
RANDOM_STATE = 42
PARAM_GRID = {
    "classifier__n_estimators": [120, 240],
    "classifier__max_depth": [3, 5],
    "classifier__learning_rate": [0.06],
}

print("Project:", PROJECT_ROOT)
print("Read-only source:", "air_quality." + SOURCE_TABLE)
print("Full feature count:", len(FULL_FEATURES))

Project: C:\Users\PLP\Documents\ChatGPT\data scie
Read-only source: air_quality.aqi_daily
Full feature count: 15


## 2. Read and validate `aqi_daily`

This cell performs a read-only query, checks required columns, and verifies one row per station and date.

In [2]:
required = ["date", "station", "station_id", *POLLUTANTS, "O3_1h_max", *WEATHER]
engine = create_engine(DB_URL, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        source = pd.read_sql_query(text(f"SELECT * FROM {SOURCE_TABLE}"), connection)
finally:
    engine.dispose()

missing = [column for column in required if column not in source.columns]
if missing:
    raise KeyError(f"{SOURCE_TABLE} is missing required columns: {missing}")
source["date"] = pd.to_datetime(source["date"], errors="coerce").dt.normalize()
if source["date"].isna().any():
    raise ValueError("aqi_daily contains invalid dates.")
if source.duplicated(["station_id", "date"]).any():
    raise ValueError("aqi_daily has duplicate station-date keys.")
for column in [*POLLUTANTS, "O3_1h_max", *WEATHER]:
    source[column] = pd.to_numeric(source[column], errors="coerce")

print(f"Rows: {len(source):,}; stations: {source.station.nunique():,}")
print(f"Date range: {source.date.min().date()} through {source.date.max().date()}")
display(source.head())

Rows: 50,278; stations: 12
Date range: 2013-03-01 through 2025-01-04


,date,station,PM2.5,PM10,SO2,NO2,CO,O3,PM2.5_valid_hours,PM10_valid_hours,...,wd_cos,temperature_hours,pressure_hours,dewpoint_hours,humidity_hours,rain_hours,wind_speed_hours,wind_direction_hours,weather_matched,weather_station_matches_pollutant
0,2013-03-01,Aotizhongxin,7.125000,10.750000,11.708333,22.583333,0.429167,74.500,24,24,...,0.874674,24,24,24,24,24,24,24,1,1
1,2013-03-01,Changping,5.083333,18.958333,16.041667,15.333333,0.387500,88.875,24,24,...,0.867750,24,24,24,24,24,24,24,1,1
2,2013-03-01,Dingling,6.375000,6.270833,3.000000,2.333333,0.204167,86.000,24,24,...,0.867750,24,24,24,24,24,24,24,1,1
3,2013-03-01,Dongsi,6.416667,9.875000,8.291667,28.229167,0.389583,88.000,24,24,...,0.874674,24,24,24,24,24,24,24,1,1
4,2013-03-01,Guanyuan,7.541667,11.666667,8.500000,28.500000,0.400000,74.000,24,24,...,0.863489,24,24,24,24,24,24,24,1,1


## 3. Rebuild AQI and exact next-day targets

AQI uses the shared pollutant breakpoints. Tomorrow's target is joined by the exact next calendar date for the same station; gaps are not treated as consecutive days.

In [3]:
daily = source.copy()
iaqi_columns = []
for pollutant in POLLUTANTS:
    values = daily[pollutant]
    iaqi_column = f"{pollutant}_iaqi"
    if pollutant == "O3":
        ozone_8h = interpolate_iaqi(values, POLLUTANT_BREAKPOINTS["O3"])
        ozone_1h = interpolate_iaqi(daily["O3_1h_max"], O3_1H_BREAKPOINTS)
        use_one_hour = values.gt(800)
        ozone_8h.loc[use_one_hour] = ozone_1h.loc[use_one_hour]
        daily[iaqi_column] = ozone_8h.round()
    else:
        daily[iaqi_column] = interpolate_iaqi(
            values, POLLUTANT_BREAKPOINTS[pollutant]
        ).round()
    iaqi_columns.append(iaqi_column)

daily["AQI"] = daily[iaqi_columns].max(axis=1, skipna=True)
today = daily[["date", "station_id", "station", *FULL_FEATURES, "AQI"]].copy()
tomorrow = daily[["date", "station_id", "AQI"]].copy()
tomorrow["date"] = tomorrow["date"] - pd.Timedelta(days=1)
tomorrow = tomorrow.rename(columns={"AQI": "target_AQI"})
data = today.merge(
    tomorrow, on=["date", "station_id"], how="inner", validate="one_to_one"
).dropna(subset=["target_AQI"]).copy()
data["target_3class"] = np.select(
    [data["target_AQI"] <= 100, data["target_AQI"] <= 200], [0, 1], default=2
).astype(int)
data["today_3class"] = np.select(
    [data["AQI"] <= 100, data["AQI"] <= 200], [0, 1], default=2
).astype(float)
data.loc[data["AQI"].isna(), "today_3class"] = np.nan
data = data.sort_values(["date", "station"]).reset_index(drop=True)

CLASS_NAMES = {
    "3-class": ["Clean / good", "Light-to-moderate polluted", "Heavily polluted"],
}
TASKS = {
    "3-class": {"target": "target_3class", "today": "today_3class", "labels": [0, 1, 2]},
}
print(f"Exact next-calendar-day examples: {len(data):,}")
counts = data["target_3class"].value_counts().reindex([0, 1, 2], fill_value=0)
display(pd.DataFrame({"class": CLASS_NAMES["3-class"], "rows": counts.to_numpy()}))

Exact next-calendar-day examples: 49,331


,class,rows
0,Clean / good,29197
1,Light-to-moderate polluted,17177
2,Heavily polluted,2957


## 4. Chronological train/test split and date-based folds

The same held-out dates and expanding-window folds are used in all experiments in this notebook.

In [4]:
unique_dates = np.array(sorted(data["date"].dropna().unique()))
if len(unique_dates) < 10:
    raise ValueError("Not enough unique dates for the chronological holdout and four time-series folds.")
cut = int(0.80 * len(unique_dates))
train_dates, test_dates = unique_dates[:cut], unique_dates[cut:]
train = data[data["date"].isin(train_dates)].copy().reset_index(drop=True)
test = data[data["date"].isin(test_dates)].copy().reset_index(drop=True)

folds = []
for earlier, later in TimeSeriesSplit(n_splits=4).split(train_dates):
    earlier_dates, later_dates = train_dates[earlier], train_dates[later]
    folds.append((
        np.flatnonzero(train["date"].isin(earlier_dates)),
        np.flatnonzero(train["date"].isin(later_dates)),
    ))
print(f"Training: {len(train):,} rows, {train.date.min().date()} to {train.date.max().date()}")
print(f"Test: {len(test):,} rows, {test.date.min().date()} to {test.date.max().date()}")
print(f"One shared chronological holdout; {len(folds)} expanding date-based folds.")

Training: 39,519 rows, 2013-03-01 to 2022-09-03
Test: 9,812 rows, 2022-09-04 to 2025-01-03
One shared chronological holdout; 4 expanding date-based folds.


## 5. Define XGBoost experiment

Pollutants only uses PM2.5, PM10, SO2, NO2, CO, and O3. Pollutants + weather uses those same six pollutant summaries and nine daily weather summaries.

In [5]:
def build_xgb_pipeline(n_classes):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", XGBClassifier(
            objective="multi:softprob", num_class=n_classes,
            eval_metric="mlogloss", random_state=RANDOM_STATE,
            n_jobs=4, tree_method="hist", verbosity=0,
        )),
    ])

def metric_row(task, model_name, y_true, prediction, cv_macro_f1=np.nan, best_params=None):
    labels = TASKS[task]["labels"]
    return {
        "task": task,
        "model": model_name,
        "cv_macro_f1": cv_macro_f1,
        "accuracy": accuracy_score(y_true, prediction),
        "macro_f1": f1_score(y_true, prediction, labels=labels, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, prediction, labels=labels, average="weighted", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "best_params": best_params if best_params is not None else "-",
    }

## 6. Tune and evaluate XGBoost by feature set

Tune one three-class model with pollutant-only inputs and one with pollutant-plus-weather inputs. Both fits use the same chronological folds and parameter grid.

In [6]:
FEATURE_SETS = {
    "Pollutants only": POLLUTANTS,
    "Pollutants + weather": FULL_FEATURES,
}
models, predictions, result_rows = {}, {}, []
for task, spec in TASKS.items():
    target_col = spec["target"]
    y_train = train[target_col].to_numpy(dtype=int)
    task_test = test.dropna(subset=[target_col]).copy()
    y_test = task_test[target_col].to_numpy(dtype=int)
    n_classes = len(spec["labels"])
    if sorted(np.unique(y_train).tolist()) != spec["labels"]:
        raise ValueError(f"{task}: training dates do not contain every expected class.")
    scorer = make_scorer(
        f1_score, labels=spec["labels"], average="macro", zero_division=0
    )
    for feature_name, feature_columns in FEATURE_SETS.items():
        search = GridSearchCV(
            build_xgb_pipeline(n_classes), PARAM_GRID, scoring=scorer,
            cv=folds, n_jobs=1, refit=True, return_train_score=False,
        )
        started = time.time()
        search.fit(train[feature_columns], y_train)
        prediction = np.asarray(search.best_estimator_.predict_proba(task_test[feature_columns])).argmax(axis=1)
        key = (task, feature_name)
        models[key] = search.best_estimator_
        predictions[key] = np.full(len(test), np.nan)
        predictions[key][test[target_col].notna().to_numpy()] = prediction
        result_rows.append(metric_row(task, feature_name, y_test, prediction,
                                      search.best_score_, search.best_params_))
        print(f"{task} | {feature_name}: CV macro-F1={search.best_score_:.4f}; test accuracy={accuracy_score(y_test, prediction):.4f}; elapsed={time.time()-started:.1f}s")
        print("Best parameters:", search.best_params_)

3-class | Pollutants only: CV macro-F1=0.5379; test accuracy=0.7679; elapsed=37.5s
Best parameters: {'classifier__learning_rate': 0.06, 'classifier__max_depth': 3, 'classifier__n_estimators': 240}


3-class | Pollutants + weather: CV macro-F1=0.6038; test accuracy=0.7882; elapsed=44.4s
Best parameters: {'classifier__learning_rate': 0.06, 'classifier__max_depth': 5, 'classifier__n_estimators': 120}


## 7. Baselines and aggregate metrics

Majority always predicts the most common training class. Persistence repeats today's class for tomorrow, using the training majority when today's AQI is unavailable.

In [7]:
# Add training-majority and persistence baselines for the three-class target.
majority_predictions = {}
for task, spec in TASKS.items():
    target_col, today_col, labels = spec["target"], spec["today"], spec["labels"]
    majority_class = int(train[target_col].value_counts().idxmax())
    majority_prediction = np.full(len(test), majority_class, dtype=int)
    persistence_prediction = test[today_col].fillna(majority_class).to_numpy(dtype=int)
    majority_predictions[(task, "Majority baseline")] = majority_prediction
    majority_predictions[(task, "Persistence baseline")] = persistence_prediction
    task_mask = test[target_col].notna().to_numpy()
    y_test = test.loc[task_mask, target_col].to_numpy(dtype=int)
    result_rows.extend([
        metric_row(task, "Majority baseline", y_test, majority_prediction[task_mask]),
        metric_row(task, "Persistence baseline", y_test, persistence_prediction[task_mask]),
    ])

results = pd.DataFrame(result_rows)
display(results.sort_values(["task", "macro_f1"], ascending=[True, False]).style.format({
    "cv_macro_f1": "{:.4f}", "accuracy": "{:.2%}", "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}", "balanced_accuracy": "{:.2%}",
}, na_rep="-"))

,task,model,cv_macro_f1,accuracy,macro_f1,weighted_f1,balanced_accuracy,best_params
1,3-class,Pollutants + weather,0.6038,78.82%,0.5317,0.7838,51.98%,"{'classifier__learning_rate': 0.06, 'classifier__max_depth': 5, 'classifier__n_estimators': 120}"
3,3-class,Persistence baseline,-,74.35%,0.5177,0.7434,51.83%,-
0,3-class,Pollutants only,0.5379,76.79%,0.4665,0.7555,46.30%,"{'classifier__learning_rate': 0.06, 'classifier__max_depth': 3, 'classifier__n_estimators': 240}"
2,3-class,Majority baseline,-,70.96%,0.2767,0.5891,33.33%,-


## 8. Per-class reports and confusion matrices

These tables show which classes are difficult and how errors are distributed.

In [8]:
for task, spec in TASKS.items():
    labels = spec["labels"]
    names = CLASS_NAMES[task]
    task_mask = test[spec["target"]].notna().to_numpy()
    y_true = test.loc[task_mask, spec["target"]].to_numpy(dtype=int)
    for feature_name in FEATURE_SETS:
        print(f"\n{task} | {feature_name}")
        y_pred = predictions[(task, feature_name)][task_mask].astype(int)
        display(pd.DataFrame(classification_report(
            y_true, y_pred, labels=labels, target_names=names,
            output_dict=True, zero_division=0,
        )).T.round(4))
        print("Confusion matrix (rows=actual, columns=predicted)")
        display(pd.DataFrame(
            confusion_matrix(y_true, y_pred, labels=labels),
            index=[f"Actual: {name}" for name in names],
            columns=[f"Predicted: {name}" for name in names],
        ))


3-class | Pollutants only


,precision,recall,f1-score,support
Clean / good,0.8174,0.8860,0.8503,6963.0000
Light-to-moderate polluted,0.6050,0.5029,0.5493,2716.0000
Heavily polluted,0.0000,0.0000,0.0000,133.0000
accuracy,0.7679,0.7679,0.7679,0.7679
macro avg,0.4741,0.4630,0.4665,9812.0000
weighted avg,0.7475,0.7679,0.7555,9812.0000


Confusion matrix (rows=actual, columns=predicted)


,Predicted: Clean / good,Predicted: Light-to-moderate polluted,Predicted: Heavily polluted
Actual: Clean / good,6169,793,1
Actual: Light-to-moderate polluted,1344,1366,6
Actual: Heavily polluted,34,99,0



3-class | Pollutants + weather


,precision,recall,f1-score,support
Clean / good,0.8510,0.8740,0.8623,6963.0000
Light-to-moderate polluted,0.6282,0.6027,0.6152,2716.0000
Heavily polluted,0.2037,0.0827,0.1176,133.0000
accuracy,0.7882,0.7882,0.7882,0.7882
macro avg,0.5609,0.5198,0.5317,9812.0000
weighted avg,0.7805,0.7882,0.7838,9812.0000


Confusion matrix (rows=actual, columns=predicted)


,Predicted: Clean / good,Predicted: Light-to-moderate polluted,Predicted: Heavily polluted
Actual: Clean / good,6086,874,3
Actual: Light-to-moderate polluted,1039,1637,40
Actual: Heavily polluted,27,95,11


## 9. Summary: does weather add value beyond pollutant readings?

The summary below reports the measured change from pollutant-only inputs to pollutant-plus-weather inputs on the same held-out station-days for the three-class AQI target.

In [9]:
print("Three-class feature-set comparison on identical test dates")
task_rows = results[results["task"] == "3-class"].set_index("model")
pollutant_only = task_rows.loc["Pollutants only"]
with_weather = task_rows.loc["Pollutants + weather"]
print(f"  Pollutants only:      accuracy={pollutant_only.accuracy:.2%}, macro-F1={pollutant_only.macro_f1:.4f}, balanced accuracy={pollutant_only.balanced_accuracy:.2%}")
print(f"  Pollutants + weather: accuracy={with_weather.accuracy:.2%}, macro-F1={with_weather.macro_f1:.4f}, balanced accuracy={with_weather.balanced_accuracy:.2%}")
print(f"  Added-weather change: accuracy={with_weather.accuracy-pollutant_only.accuracy:+.2%}, macro-F1={with_weather.macro_f1-pollutant_only.macro_f1:+.4f}")

print("\nInterpretation")
print("- A positive added-weather change means weather improved that metric on this holdout; a negative change means it did not.")
print("- Both models use today's pollutant readings. Only the second receives today's weather summaries.")
print("- The comparison shows predictive value for these dates and stations, not a causal effect of weather or guaranteed gains elsewhere.")
print("- Compare class-level reports, especially heavy-pollution recall and false alarms, before choosing a model.")

Three-class feature-set comparison on identical test dates
  Pollutants only:      accuracy=76.79%, macro-F1=0.4665, balanced accuracy=46.30%
  Pollutants + weather: accuracy=78.82%, macro-F1=0.5317, balanced accuracy=51.98%
  Added-weather change: accuracy=+2.03%, macro-F1=+0.0652

Interpretation
- A positive added-weather change means weather improved that metric on this holdout; a negative change means it did not.
- Both models use today's pollutant readings. Only the second receives today's weather summaries.
- The comparison shows predictive value for these dates and stations, not a causal effect of weather or guaranteed gains elsewhere.
- Compare class-level reports, especially heavy-pollution recall and false alarms, before choosing a model.
